In [2]:
from googleapiclient.discovery import build
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from typing import Any, Dict, List, Optional, Tuple
import base64
import json
from datetime import datetime

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
# creds 是你授权后保存 token 的逻辑里得到的凭证
creds = Credentials.from_authorized_user_file("token.json", SCOPES)
service = build("gmail", "v1", credentials=creds)

def _get_header(headers: List[Dict[str, str]], name: str) -> Optional[str]:
    for h in headers:
        if h.get("name", "").lower() == name.lower():
            return h.get("value")
    return None


def _urlsafe_b64decode(data: str) -> str:
    # Gmail 返回的 body.data 是 urlsafe base64
    missing_padding = (-len(data)) % 4
    if missing_padding:
        data += "=" * missing_padding
    raw = base64.urlsafe_b64decode(data.encode("utf-8"))
    # 尽量用 utf-8 解码；失败就用 replace 保底
    return raw.decode("utf-8", errors="replace")


def _extract_bodies_from_payload(payload: Dict[str, Any]) -> Tuple[str, str]:
    """
    从 Gmail message payload 里递归提取:
    - text/plain 合并为一个字符串
    - text/html 合并为一个字符串
    """
    text_parts: List[str] = []
    html_parts: List[str] = []

    def walk(part: Dict[str, Any]) -> None:
        mime_type = part.get("mimeType", "")
        body = part.get("body", {}) or {}
        data = body.get("data")

        # 有些邮件正文可能直接在该 part 的 body.data 里
        if data:
            decoded = _urlsafe_b64decode(data)
            if mime_type == "text/plain":
                text_parts.append(decoded)
            elif mime_type == "text/html":
                html_parts.append(decoded)

        # multipart 继续递归
        for sub in part.get("parts", []) or []:
            walk(sub)

    walk(payload)

    return ("\n".join(t.strip() for t in text_parts if t.strip()),
            "\n".join(h.strip() for h in html_parts if h.strip()))


def _extract_attachments_metadata(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    仅保存附件元信息（不下载内容），例如 filename、mimeType、size、attachmentId。
    """
    attachments: List[Dict[str, Any]] = []

    def walk(part: Dict[str, Any]) -> None:
        filename = part.get("filename")
        body = part.get("body", {}) or {}
        attachment_id = body.get("attachmentId")
        mime_type = part.get("mimeType", "")

        if filename and attachment_id:
            attachments.append({
                "filename": filename,
                "mimeType": mime_type,
                "size": body.get("size"),
                "attachmentId": attachment_id,
            })

        for sub in part.get("parts", []) or []:
            walk(sub)

    walk(payload)
    return attachments


def archive_latest_emails(
    max_results: int = 10,
    output_path: str = "gmail_archive.json",
) -> None:
    results = service.users().messages().list(
        userId="me",
        maxResults=max_results,
    ).execute()

    messages = results.get("messages", []) or []
    archived: List[Dict[str, Any]] = []

    for m in messages:
        msg_id = m["id"]
        msg_data = service.users().messages().get(
            userId="me",
            id=msg_id,
            format="full",
        ).execute()

        payload = msg_data.get("payload", {}) or {}
        headers = payload.get("headers", []) or []

        subject = _get_header(headers, "Subject")
        from_addr = _get_header(headers, "From")
        to_addr = _get_header(headers, "To")
        cc_addr = _get_header(headers, "Cc")
        bcc_addr = _get_header(headers, "Bcc")
        date_hdr = _get_header(headers, "Date")

        text_body, html_body = _extract_bodies_from_payload(payload)
        attachments = _extract_attachments_metadata(payload)

        archived.append({
            "id": msg_data.get("id"),
            "threadId": msg_data.get("threadId"),
            "labelIds": msg_data.get("labelIds", []),
            "internalDate_ms": msg_data.get("internalDate"),
            "snippet": msg_data.get("snippet"),

            "headers": {
                "from": from_addr,
                "to": to_addr,
                "cc": cc_addr,
                "bcc": bcc_addr,
                "subject": subject,
                "date": date_hdr,
            },

            "body": {
                "text": text_body,
                "html": html_body,
            },

            "attachments": attachments,
        })

    out = {
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "max_results": max_results,
        "count": len(archived),
        "messages": archived,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"✅ Archived {len(archived)} emails -> {output_path}")


if __name__ == "__main__":
    archive_latest_emails(max_results=10, output_path="gmail_archive.json")

✅ Archived 10 emails -> gmail_archive.json
